# Lab 2 -Part B


### Task 0 - Generating the Data


| File | Rows | Size | Data type |
|---|---:|---:|---|
| trips.csv | 1,480,546 | 151.1 MB | Structured |
| gps_pings.jsonl | 484,433 | 91.6 MB | Semi-structured |
| reviews.jsonl | 20,000 | 1.8 MB | Unstructured |

- **Structured:** trips.csv, b/c it has a fixed table structure with rows and columns.
- **Semi-structured:** gps_pings.jsonl, because it contains JSON records with nested fields.
- **Unstructured:** reviews.jsonl, because it contains free-text reviews.


### Task 1 – Volume: How Much Memory Does the Data Need?

In [ ]:
import os
import time
import json
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", None)

mb = lambda df: df.memory_usage(deep=True).sum() / 1e6
t = time.perf_counter()
trips = pd.read_csv("raw/trips.csv")

print(
    f"rows {len(trips):,}; "
    f"file {os.path.getsize('raw/trips.csv') / 1e6:.1f} MB; "
    f"read in {time.perf_counter() - t:.1f} s; "
    f"in memory {mb(trips):.1f} MB")

print(trips.dtypes.value_counts().to_dict())

small = trips.astype({
    "pickup_zone": "category", "dropoff_zone": "category",
    "status": "category", "vehicle": "category",
    "payment": "category", "trip_id": "int32",  "driver_id": "int16",
    "rider_id": "int32",
    "distance_km": "float32",  "duration_min": "float32",
    "fare_birr": "float32",  "surge": "float32"})

small["request_time"] = pd.to_datetime(small.request_time)
print(f"with smaller types: {mb(small):.1f} MB")


t = time.perf_counter()
revenue, counts = 0.0, pd.Series(dtype="int64")

for chunk in pd.read_csv(
    "raw/trips.csv",
    usecols=["status", "fare_birr"],
    chunksize=200_000):
    revenue += chunk.fare_birr.sum()
    counts = counts.add(
        chunk.status.value_counts(),
        fill_value=0
    )

print(
    f"chunked: revenue {revenue:,.0f} birr, "
    f"{counts.astype(int).to_dict()} "
    f"in {time.perf_counter() - t:.1f} s"
)

print("same as full load:", revenue == trips.fare_birr.sum())


t = time.perf_counter()
revenue, counts = 0.0, pd.Series(dtype="int64")

for chunk in pd.read_csv(
    "raw/trips.csv",
    usecols=["status", "fare_birr"],
    chunksize=200_000):
    revenue += chunk.fare_birr.sum()
    counts = counts.add(
        chunk.status.value_counts(),
        fill_value=0
    )

print(
    f"chunked: revenue {revenue:,.0f} birr, "
    f"{counts.astype(int).to_dict()} "
    f"in {time.perf_counter() - t:.1f} s"
)

print("same as full load:", revenue == trips.fare_birr.sum())


rows 1,480,546; file 151.1 MB; read in 6.2 s; in memory 602.9 MB
{dtype('O'): 6, dtype('float64'): 4, dtype('int64'): 3}
with smaller types: 57.7 MB
chunked: revenue 317,922,223 birr, {'cancelled': 74133, 'completed': 1315214, 'no_driver': 91199} in 3.3 s
same as full load: True
chunked: revenue 317,922,223 birr, {'cancelled': 74133, 'completed': 1315214, 'no_driver': 91199} in 3.5 s
same as full load: True


### Task 1 Results


**1.**
- File size: 151.1 MB
- Read time: 6.2 seconds
- Memory used: 602.9 MB
- Memory after changing data types: 57.7 MB
- Chunked processing:
  - Total revenue: 317,922,223 birr
  - Completed trips: 1,315,214
  - Cancelled trips: 74,133
  - Trips with no driver: 91,199
- The chunked result matched the full dataset result.

**2. Why does the table take about four times its file size in memory? Why does the category type save so much?**

The table uses more memory because pandas stores data in memory using additional overhead, and some columns are stored as larger data types than necessary. Text columns also require extra memory. The category type saves memory by storing repeated values as codes instead of storing the same text many times.

**3. Estimate the memory needed for two years of trips at the same rate, with and without smaller types. When would chunked processing or a cluster become necessary?**

The dataset covers 60 days. Two years is approximately 12.17 times longer.

- Without smaller types, approximately 7.34 GB.
- With smaller types, approximately 0.70 GB.

Chunked processing becomes necessary when the dataset is too large to load into available memory. A cluster becomes necessary when the data or processing workload exceeds the capacity of one computer.

## Task 2 - Storage Formats: CSV, Compressed CSV and Parquet

In [ ]:
os.makedirs("lake", exist_ok=True)

small.to_parquet("lake/trips.parquet", index=False)

t = time.perf_counter()
trips.to_csv("lake/trips.csv.gz", index=False)
gz_write = time.perf_counter() - t

for path in [
    "raw/trips.csv",
    "lake/trips.csv.gz",
    "lake/trips.parquet"
]:
    print(f"{path:<20} {os.path.getsize(path) / 1e6:7.1f} MB")

print(f"gzip CSV took {gz_write:.1f} s to write")

def timed(label, fn):
    t = time.perf_counter()
    df = fn()
    print(f"{label:<34} {time.perf_counter() - t:5.2f} s {df.shape}")

for _ in range(2):
    timed(
        "CSV, all columns",
        lambda: pd.read_csv("raw/trips.csv"))
    timed(
        "gzip CSV, all columns",
        lambda: pd.read_csv("lake/trips.csv.gz")
    )
    timed(
        "Parquet, all columns",
        lambda: pd.read_parquet("lake/trips.parquet"))
    timed(
        "CSV, 2 columns",
        lambda: pd.read_csv(
            "raw/trips.csv",
            usecols=["pickup_zone", "fare_birr"]))
    timed(
        "Parquet, 2 columns",
        lambda: pd.read_parquet(
            "lake/trips.parquet",
            columns=["pickup_zone", "fare_birr"]))

back = pd.read_parquet("lake/trips.parquet")
print(
    "types kept by Parquet:",
    back.dtypes[
        ["request_time", "pickup_zone", "trip_id"]
    ].astype(str).to_dict())

raw/trips.csv          151.1 MB
lake/trips.csv.gz       32.1 MB
lake/trips.parquet      32.3 MB
gzip CSV took 41.9 s to write
CSV, all columns                    6.45 s (1480546, 13)
gzip CSV, all columns               8.36 s (1480546, 13)
Parquet, all columns                3.22 s (1480546, 13)
CSV, 2 columns                      3.16 s (1480546, 2)
Parquet, 2 columns                  0.10 s (1480546, 2)
CSV, all columns                    5.70 s (1480546, 13)
gzip CSV, all columns               8.31 s (1480546, 13)
Parquet, all columns                0.22 s (1480546, 13)
CSV, 2 columns                      2.91 s (1480546, 2)
Parquet, 2 columns                  0.11 s (1480546, 2)
types kept by Parquet: {'request_time': 'datetime64[ns]', 'pickup_zone': 'category', 'trip_id': 'int32'}


### Task 2 Result

**1**

File sizes:

| File | Size |
|---|---:|
| Raw CSV | 151.1 MB |
| Gzip CSV | 32.1 MB |
| Parquet | 32.3 MB |

Gzip CSV write time: **41.9 seconds**

Read times:

| Read operation | First run | Second run |
|---|---:|---:|
| CSV, all columns | 6.45 s | 5.70 s |
| Gzip CSV, all columns | 8.36 s | 8.31 s |
| Parquet, all columns | 3.22 s | 0.22 s |
| CSV, 2 columns | 3.16 s | 2.91 s |
| Parquet, 2 columns | 0.10 s | 0.11 s |

**2. Gzip CSV and Parquet are about the same size. Why is Parquet so much faster to read, especially for two columns? What does Parquet keep that CSV loses?**

Parquet is faster because it uses columnar storage, allowing it to read only the columns needed instead of processing the entire dataset.

Parquet also preserves data types and schema information, such as datetime, category and integer types. CSV stores data as plain text and does not preserve these data types.



## Task 3 — Operational versus Analytical: SQLite and DuckDB

In [ ]:
import sqlite3, duckdb

if os.path.exists("lake/trips.db"):
    os.remove("lake/trips.db")

con = sqlite3.connect("lake/trips.db")
t = time.perf_counter()
trips.to_sql("trips", con, index=False)
con.execute("CREATE UNIQUE INDEX idx_trip ON trips(trip_id)")

print(f"SQLite load and index: {time.perf_counter() - t:.1f} s; file {os.path.getsize('lake/trips.db') / 1e6:.0f} MB")

ids = np.random.default_rng(0).integers(1, len(trips) + 1, 1000)
t = time.perf_counter()

for i in ids:
    con.execute(
        "SELECT status, fare_birr FROM trips WHERE trip_id = ?",
        (int(i),)
    ).fetchone()

print(f"SQLite: 1,000 lookups by trip_id, {(time.perf_counter() - t) * 1000 / 1000:.3f} ms each")

t = time.perf_counter()
con.execute(
    "UPDATE trips SET status = 'cancelled', fare_birr = 0 WHERE trip_id = 42")
con.commit()

print(f"SQLite: update one trip and commit, {(time.perf_counter() - t) * 1000:.1f} ms")

Q = """SELECT pickup_zone, COUNT(*) AS requests,
              AVG(CASE WHEN status = 'no_driver' THEN 1.0 ELSE 0 END) AS no_driver_rate,
              SUM(fare_birr) AS revenue
       FROM {src} GROUP BY pickup_zone ORDER BY revenue DESC"""

t = time.perf_counter()
a = pd.read_sql(Q.format(src="trips"), con)
t_sqlite = time.perf_counter() - t

duck = duckdb.connect()
t = time.perf_counter()
b = duck.sql(Q.format(src="'lake/trips.parquet'")).df()
t_duck = time.perf_counter() - t

t = time.perf_counter()
duck.sql(Q.format(src="'lake/trips.parquet'")).df()
t_duck2 = time.perf_counter() - t

print(f"revenue by zone: SQLite {t_sqlite:.2f} s, DuckDB on Parquet {t_duck:.2f} s (again {t_duck2:.2f} s)")
print(b.round(3).to_string(index=False))
print("same answer apart from trip 42:", (a.requests.values == b.requests.values).all())

SQLite load and index: 17.6 s; file 174 MB
SQLite: 1,000 lookups by trip_id, 0.400 ms each
SQLite: update one trip and commit, 365.5 ms
revenue by zone: SQLite 3.85 s, DuckDB on Parquet 2.48 s (again 0.09 s)
     pickup_zone  requests  no_driver_rate    revenue
            Bole    296144           0.097 63095267.0
          Kirkos    206988           0.097 44472229.0
           Arada    162998           0.040 35015461.0
            Yeka    133406           0.040 28646605.0
   Kolfe Keranio    118842           0.039 25638186.0
Nifas Silk-Lafto    118426           0.040 25531942.0
    Addis Ketema    118264           0.041 25389766.0
          Lideta    103646           0.040 22289818.0
          Gulele     88678           0.040 19128469.0
    Akaky Kaliti     74058           0.041 15977014.0
       Lemi Kura     59096           0.098 12737466.0
same answer apart from trip 42: True


### Task 3 Results

**1**
- SQLite load and index: 17.6 seconds (174 MB).
- 1,000 lookups: 0.400 ms each.
- Update one trip: 365.5 ms.
- Revenue query:
  - SQLite: 3.85 seconds.
  - DuckDB: 2.48 seconds (0.09 seconds on the second run).

| Pickup zone | Requests | No-driver rate | Revenue (birr) |
|---|---:|---:|---:|
| Bole | 296,144 | 0.097 | 63,095,267 |
| Kirkos | 206,988 | 0.097 | 44,472,229 |
| Arada | 162,998 | 0.040 | 35,015,461 |
| Yeka | 133,406 | 0.040 | 28,646,605 |
| Kolfe Keranio | 118,842 | 0.039 | 25,638,186 |
| Nifas Silk-Lafto | 118,426 | 0.040 | 25,531,942 |
| Addis Ketema | 118,264 | 0.041 | 25,389,766 |
| Lideta | 103,646 | 0.040 | 22,289,818 |
| Gulele | 88,678 | 0.040 | 19,128,469 |
| Akaky Kaliti | 74,058 | 0.041 | 15,977,014 |
| Lemi Kura | 59,096 | 0.098 | 12,737,466 |

**2. why SQLite finds one trip so quickly but is slower than DuckDB for the revenue table. Which engine suits an operational workload, and which an analytical one?**

SQLite uses an index to find an individual trip quickly. DuckDB is faster for analytical queries because it is designed to process large amounts of data efficiently.
SQLite is suitable for operational workloads, while DuckDB is suitable for analytical workloads.

**3. Three sub-cities that have a no-driver rate more than twice the others?**

Bole, Kirkos, and Lemi Kura.


## Task 4 - Velocity: Processing Pings as a Stream

In [ ]:
from collections import defaultdict
from datetime import datetime

WATERMARK = 30  
counts, closed_until, late, seen, max_ts = defaultdict(int), None, 0, 0, None
delays = []

t = time.perf_counter()

with open("raw/gps_pings.jsonl") as f:
    for line in f:
        ev = json.loads(line)
        ts = datetime.fromisoformat(ev["ts"])

        delays.append(
            (datetime.fromisoformat(ev["received_at"]) - ts).total_seconds()
        )

        max_ts = ts if max_ts is None or ts > max_ts else max_ts
        window = ts.replace(second=0)
        watermark = max_ts.timestamp() - WATERMARK

        if window.timestamp() + 60 <= watermark:
            late += 1
            continue

        counts[(window, ev["status"])] += 1
        seen += 1

elapsed = time.perf_counter() - t

print(
    f"stream: {seen + late:,} pings in {elapsed:.1f} s = "
    f"{(seen + late) / elapsed:,.0f} pings/s; "
    f"dropped as late: {late:,}")

d = np.array(delays)

print(
    f"delay received - event: median {np.median(d):.0f} s, "
    f"99th percentile {np.percentile(d, 99):.0f} s, "
    f"over 30 s: {(d > 30).mean():.2%}")

batch = pd.read_json("raw/gps_pings.jsonl", lines=True)
batch["window"] = pd.to_datetime(batch.ts).dt.floor("min")
bc = batch.groupby(["window", "status"]).size()

sc = pd.Series(counts)
sc.index = pd.MultiIndex.from_tuples(
    sc.index, names=["window", "status"]
)

print(
    f"batch total {bc.sum():,}; stream total {sc.sum():,}; "
    f"difference {bc.sum() - sc.sum():,}")

busiest = bc.unstack().sum(axis=1).idxmax()
print("busiest minute:", busiest, bc.unstack().loc[busiest].to_dict())

stream: 484,433 pings in 11.7 s = 41,489 pings/s; dropped as late: 4,046
delay received - event: median 3 s, 99th percentile 105 s, over 30 s: 1.99%
batch total 484,433; stream total 480,387; difference 4,046
busiest minute: 2026-10-15 12:45:00 {'available': 362.0, 'on_trip': 429.0}



### Task 4 Report

**1.**

- Throughput: 41,489 pings/second.
- Late pings dropped: 4,046.
- Median delay: 3 seconds.
- 99th percentile delay: 105 seconds.
- Pings delayed over 30 seconds: 1.99%.
- Batch total: 484,433.
- Stream total: 480,387.
- Difference: 4,046.
- Busiest minute: 12:45, with 362 available and 429 on-trip pings.

**2. Comparing measured throughput with the real load. About how many drivers pinging every 4 seconds could one core handle?**

The real load is 1,000 pings per minute, or about 17 pings per second. Our measured throughput of 41,489 pings per second is much higher.
At one ping every 4 seconds, one core could theoretically handle approximately 165,956 drivers.

**3. The 30-second watermark. What would change if it were 3 minutes?**

The 30-second watermark allows late pings to arrive before their time window is closed. Pings arriving after the window closes are dropped.
A 3-minute watermark would allow more late pings to be included, reducing dropped records, but results would take longer to finalize and require more memory.


## Task 5 - Variety: Nested JSON, a Changing Schema and Free Text

In [ ]:
with open("raw/gps_pings.jsonl") as f:
    pings = pd.json_normalize([json.loads(line) for line in f])

print("columns:", list(pings.columns))

print(
    "missing values per column:",
    pings.isna().sum()[lambda s: s > 0].to_dict()
)

print(
    pings.groupby("app.version").battery
    .apply(lambda s: s.isna().mean())
    .rename("battery missing").to_dict()
)

reviews = pd.read_json("raw/reviews.jsonl", lines=True)

reviews["amharic"] = reviews.text.str.contains(r"[ሀ-፿]")

print(
    f"reviews {len(reviews):,}; "
    f"with Amharic text {reviews.amharic.mean():.1%}; "
    f"characters: median {reviews.text.str.len().median():.0f}, "
    f"bytes in UTF-8 of 'ጥሩ ሹፌር': "
    f"{len('ጥሩ ሹፌር'.encode())}"
)

topics = {
    "late": "late|waited|took", "price": "expensive|price",
    "app": "app", "rude": "rude|phone",
    "clean": "clean",
    "safe": "safe|careful|knew the road", "amharic": r"[ሀ-፿]"
}

for k, pat in topics.items():
    reviews[k] = reviews.text.str.contains(pat, case=False)

r = reviews.merge(
    trips[["trip_id", "surge", "fare_birr", "pickup_zone"]],
    on="trip_id"
)

summary = pd.DataFrame({
    k: {
        "reviews": int(r[k].sum()),
        "avg rating": r.loc[r[k], "rating"].mean(),
        "avg fare": r.loc[r[k], "fare_birr"].mean()
    }
    for k in topics
}).T

print(summary.round(2))

print(
    "share of reviews about price:",
    r.groupby(r.surge > 1).price.mean()
    .rename({False: "no surge", True: "surge"})
    .round(3).to_dict())

columns: ['driver_id', 'ts', 'lat', 'lon', 'status', 'received_at', 'app.version', 'app.os', 'battery']
missing values per column: {'lat': 2405, 'lon': 2405, 'battery': 343943}
{'4.7': 1.0, '5.0': 0.0}
reviews 20,000; with Amharic text 11.1%; characters: median 25, bytes in UTF-8 of 'ጥሩ ሹፌር': 16
         reviews  avg rating  avg fare
late      3370.0        1.99    239.19
price     4058.0        2.51    255.71
app       1526.0        2.02    237.55
rude      1280.0        1.49    235.65
clean     3787.0        4.49    239.30
safe      3755.0        4.49    234.54
amharic   2224.0        4.49    241.28
share of reviews about price: {'no surge': 0.148, 'surge': 0.327}


### Task 5 Report

**1.**

The GPS data contains these columns: `driver_id`, `ts`, `lat`, `lon`, `status`, `received_at`, `app.version`, `app.os` and `battery`.

Missing values:
- Latitude: 2,405
- Longitude: 2,405
- Battery: 343,943

Battery missing by app version:
- Version 4.7: 100%
- Version 5.0: 0%

The schema changed when app version 5.0 introduced battery information. Missing battery values should be treated as unavailable, not as zero.

**2. Report the Amharic share and UTF-8 byte count. Why does 'ጥሩ ሹፌር' take 16 bytes for 6 characters?**

- Reviews containing Amharic: 11.1%.
- Median review length: 25 characters.
- UTF-8 size of 'ጥሩ ሹፌር': 16 bytes.

It takes 16 bytes because each of the five Amharic characters uses 3 bytes, while the space uses 1 byte.

**3. The topic table and price-complaint shares. What to tell to the pricing team?**

| Topic | Reviews | Average rating | Average fare (birr) |
|---|---:|---:|---:|
| Late | 3,370 | 1.99 | 239.19 |
| Price | 4,058 | 2.51 | 255.71 |
| App | 1,526 | 2.02 | 237.55 |
| Rude | 1,280 | 1.49 | 235.65 |
| Clean | 3,787 | 4.49 | 239.30 |
| Safe | 3,755 | 4.49 | 234.54 |
| Amharic | 2,224 | 4.49 | 241.28 |

Price complaints:
- No surge: 14.8%.
- Surge: 32.7%.

**The pricing team should investigate surge pricing because price complaints are more than twice as common during surge periods.**


## Task 6 -  Veracity: Checks on the GPS Pings

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    p1, p2 = np.radians(lat1), np.radians(lat2)
    a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

p = pings.copy()
p["ts"] = pd.to_datetime(p.ts)

checks = {
    "rows": len(p),
    "missing position": int(p.lat.isna().sum()),
    "position (0, 0)": int(((p.lat == 0) & (p.lon == 0)).sum()),
    "outside Addis Ababa box": int((p.lat.notna() & ~(p.lat.between(8.80, 9.15) & p.lon.between(38.60, 38.95))).sum()),
    "exact duplicates": int(p.duplicated().sum())}
p = p.drop_duplicates()
p = p[p.lat.between(8.80, 9.15) & p.lon.between(38.60, 38.95)].sort_values(["driver_id", "ts"])

prev = p.groupby("driver_id")[["lat", "lon", "ts"]].shift()

p["speed_kmh"] = haversine_km(
    prev.lat, prev.lon, p.lat, p.lon
) / ((p.ts - prev.ts).dt.total_seconds() / 3600)

checks["speed over 120 km/h"] = int((p.speed_kmh > 120).sum())
checks["clean rows"] = int((p.speed_kmh.fillna(0) <= 120).sum())
print(pd.Series(checks).to_string())

print(
    "speed percentiles (km/h):",
    p.speed_kmh.quantile([0.5, 0.99, 0.999]).round(1).to_dict())
clean_pings = p[p.speed_kmh.fillna(0) <= 120]

rows                       484433
missing position             2405
position (0, 0)               486
outside Addis Ababa box       546
exact duplicates             4854
speed over 120 km/h          2733
clean rows                 473920
speed percentiles (km/h): {0.5: 9.4, 0.99: 26.4, 0.999: 553.4}


### Task 6 Report

**1.checks and the speed percentiles.**

| Check | Result |
|---|---:|
| Total rows | 484,433 |
| Missing positions | 2,405 |
| Positions at (0, 0) | 486 |
| Outside Addis Ababa bounding box | 546 |
| Exact duplicates | 4,854 |
| Speed over 120 km/h | 2,733 |
| Clean rows | 473,920 |

Speed percentiles:
- 50th percentile: 9.4 km/h
- 99th percentile: 26.4 km/h
- 99.9th percentile: 553.4 km/h

**2. How does the speed check find GPS jumps that the city box misses? Why are about two rows flagged for each jump?**

The city box only checks whether coordinates are inside the expected area. The speed check compares consecutive GPS positions for each driver and detects unrealistic movement between them, even when both positions are inside the city box.

A jump can flag two rows because the unrealistic movement affects the transition from one point to the next, so both points may be associated with the detected jump.

**3. For each problem, would you drop, repair or keep and flag rows? Give a reason.**

- Missing positions: keep and flag because the other information may still be useful.
- Positions at (0, 0): drop or mark invalid because they are not realistic Addis Ababa locations.
- Outside-city positions: flag and investigate; drop if confirmed invalid.
- Exact duplicates: drop because they repeat the same record.
- Unrealistic speeds: flag and investigate because they may indicate GPS errors or jumps.


## Task 7 -  A Small Data Lake, and the Value of a Decision

In [14]:

import shutil

shutil.rmtree("lake/clean", ignore_errors=True)
shutil.rmtree("lake/curated", ignore_errors=True)

small.assign(
    month=small.request_time.dt.month
).to_parquet(
    "lake/clean/trips",
    partition_cols=["month"],
    index=False)

clean_pings.assign(
    hour=clean_pings.ts.dt.hour
).to_parquet(
    "lake/clean/pings",
    partition_cols=["hour"],
    index=False)

os.makedirs("lake/curated", exist_ok=True)

zone_hour = duck.sql("""
    SELECT pickup_zone, hour(request_time) AS hour, COUNT(*) AS requests,
           SUM(CASE WHEN status = 'no_driver' THEN 1 ELSE 0 END) AS no_driver,
           AVG(CASE WHEN status = 'completed' THEN fare_birr END) AS avg_fare
    FROM read_parquet('lake/clean/trips/*/*.parquet', hive_partitioning = true)
    GROUP BY ALL
""").df()

zone_hour.to_parquet(
    "lake/curated/zone_hour.parquet",
    index=False)

for layer in ["raw", "lake/clean", "lake/curated"]:
    files = [
        os.path.join(d, f)
        for d, _, fs in os.walk(layer)
        for f in fs]

    print(
        f"{layer:<14} {len(files):>3} files "
        f"{sum(map(os.path.getsize, files)) / 1e6:9.3f} MB")

zone_hour["no_driver_rate"] = (
    zone_hour.no_driver / zone_hour.requests)

print(
    zone_hour.sort_values(
        "no_driver_rate",
        ascending=False
    ).head(6).round(3).to_string(index=False))

raw              3 files   244.536 MB
lake/clean      26 files    56.057 MB
lake/curated     1 files     0.009 MB
pickup_zone  hour  requests  no_driver  avg_fare  no_driver_rate
  Lemi Kura    18      5382     1241.0   355.480           0.231
  Lemi Kura    17      4977     1125.0   352.502           0.226
     Kirkos     8     16283     3657.0   352.292           0.225
       Bole    18     26165     5813.0   349.377           0.222
       Bole     8     23161     5117.0   351.402           0.221
       Bole    17     24478     5397.0   349.825           0.220


### Task 7 Report
**1. File counts and total sizes for each layer.**

| Layer | Number of files | Total size |
|---|---:|---:|
| Raw | 3 | 244.536 MB |
| Clean | 26 | 56.057 MB |
| Curated | 1 | 0.009 MB |

**2.The six zone-hour combinations with the highest no-driver rates.**

| Pickup zone | Hour | Requests | No-driver trips | Average fare (birr) | No-driver rate |
|---|---:|---:|---:|---:|---:|
| Lemi Kura | 18 | 5,382 | 1,241 | 355.480 | 0.231 |
| Lemi Kura | 17 | 4,977 | 1,125 | 352.502 | 0.226 |
| Kirkos | 8 | 16,283 | 3,657 | 352.292 | 0.225 |
| Bole | 18 | 26,165 | 5,813 | 349.377 | 0.222 |
| Bole | 8 | 23,161 | 5,117 | 351.402 | 0.221 |
| Bole | 17 | 24,478 | 5,397 | 349.825 | 0.220 |
